In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/Users/renangomes/Desktop/political-ads-dashboard/data/Global_Cybersecurity_Threats_2015-2024.csv', encoding = 'UTF-8').copy()

df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe().round(2)

In [ ]:
# Célula 4 — Renomear colunas para snake_case
df = df.rename(columns={
    'Country': 'country',
    'Year': 'year',
    'Attack Type': 'attack_type',
    'Target Industry': 'target_industry',
    'Financial Loss (in Million $)': 'financial_loss',
    'Number of Affected Users': 'affected_users',
    'Attack Source': 'attack_source',
    'Security Vulnerability Type': 'vulnerability_type',
    'Defense Mechanism Used': 'defense_mechanism',
    'Incident Resolution Time (in Hours)': 'resolution_time_hours'
})

In [ ]:
#Conversão para category
cols = ['country', 'attack_type', 'target_industry', 'attack_source', 'vulnerability_type', 'defense_mechanism']

for col in cols:
    df.loc[:, col] = df[col].astype('category')

print(df.dtypes)

In [ ]:
# Criação da coluna sevetity com os quartis da coluna financial loss
df.loc[:, 'severity'] = pd.qcut(
    df['financial_loss'],
    q = 3,
    labels = ['baixo', 'médio', 'alto']
)

print(df['severity'].value_counts())

In [ ]:
df.loc[:, 'response_speed'] = pd.cut(df['resolution_time_hours'], bins = [0,24,48,72], labels = ['rápido', 'médio','lento'])

In [17]:
df.describe(include = 'all')

,country,year,attack_type,target_industry,financial_loss,affected_users,attack_source,vulnerability_type,defense_mechanism,resolution_time_hours,severity,response_speed
count,3000,3000.000000,3000,3000,3000.000000,3000.000000,3000,3000,3000,3000.000000,3000,3000
unique,10,NaN,6,7,NaN,NaN,4,4,5,NaN,3,3
top,UK,NaN,DDoS,IT,NaN,NaN,Nation-state,Zero-day,Antivirus,NaN,baixo,médio
freq,321,NaN,531,478,NaN,NaN,794,785,628,NaN,1000,1026
mean,NaN,2019.570333,NaN,NaN,50.492970,504684.136333,NaN,NaN,NaN,36.476000,NaN,NaN
std,NaN,2.857932,NaN,NaN,28.791415,289944.084972,NaN,NaN,NaN,20.570768,NaN,NaN
min,NaN,2015.000000,NaN,NaN,0.500000,424.000000,NaN,NaN,NaN,1.000000,NaN,NaN
25%,NaN,2017.000000,NaN,NaN,25.757500,255805.250000,NaN,NaN,NaN,19.000000,NaN,NaN
50%,NaN,2020.000000,NaN,NaN,50.795000,504513.000000,NaN,NaN,NaN,37.000000,NaN,NaN
75%,NaN,2022.000000,NaN,NaN,75.630000,758088.500000,NaN,NaN,NaN,55.000000,NaN,NaN


País mais atacado: UK (321x)
Tipo de ataque mais usado: DDoS (531x)
Industria mais atacada: IT (478)
Mecanismo de defesa mais usado: Antivírus



In [24]:
# Ataques por ano
ataques_por_ano = df.groupby('year').size().reset_index(name = 'total_ataques')

print(ataques_por_ano)


   year  total_ataques
0  2015            277
1  2016            285
2  2017            319
3  2018            310
4  2019            263
5  2020            315
6  2021            299
7  2022            318
8  2023            315
9  2024            299


In [36]:
prejuizo_pais = df.groupby('country')['financial_loss'].agg(['mean', 'sum', 'count']).round(2)

prejuizo_pais.columns = ['média', 'total', 'quantidade_ataques']

prejuizo_pais = prejuizo_pais.sort_values('total', ascending = False)

print(prejuizo_pais)

           média     total  quantidade_ataques
country                                       
UK         51.41  16502.99                 321
Germany    54.27  15793.24                 291
Brazil     50.91  15782.62                 310
Australia  51.86  15403.00                 297
Japan      49.83  15197.34                 305
France     49.09  14972.28                 305
USA        51.61  14812.12                 287
Russia     49.95  14734.73                 295
India      47.29  14566.12                 308
China      48.81  13714.47                 281


In [ ]:
#País com maior prejuízo: UK
#País com maior média de ataques: Germany

In [32]:
cross = pd.crosstab(df['attack_type'], df['target_industry'])

print(cross)

target_industry    Banking  Education  Government  Healthcare  IT  Retail  \
attack_type                                                                 
DDoS                    71         73          71          78  91      62   
Malware                 61         70          64          81  67      68   
Man-in-the-Middle       77         65          53          58  80      70   
Phishing                96         73          68          63  89      89   
Ransomware              69         71          72          77  74      71   
SQL Injection           71         67          75          72  77      63   

target_industry    Telecommunications  
attack_type                            
DDoS                               85  
Malware                            74  
Man-in-the-Middle                  56  
Phishing                           51  
Ransomware                         59  
SQL Injection                      78  


In [35]:
defesa_efic = df.groupby('defense_mechanism')['resolution_time_hours'].mean().sort_values()

print(defesa_efic)

defense_mechanism
Firewall              35.714530
Antivirus             36.573248
Encryption            36.589527
AI-based Detection    36.612350
VPN                   36.864379
Name: resolution_time_hours, dtype: float64
